# 1. Training data exploration

Inspect the saved training-image audit, class balance and exact duplicates. Inputs are the original training manifest and saved audit reports. Only training-image metadata is used by default. Report figures remain in `outputs/figures/`; an optional new audit writes separately under `outputs/reproduction/`.

File hashes identify identical encoded files. Pixel hashes include decoded RGB pixels and dimensions, so differently encoded copies can also match. These methods do not identify every near-duplicate or related crop. Numeric filenames do not establish animal or source groups.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import torch

# Set this explicitly on Kaggle to the extracted project folder.
PROJECT_ROOT = None
if PROJECT_ROOT is None:
    PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                         if (p / "src/models.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the extracted project containing src/ and data/processed/")
PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.project import RUNS, load_metadata, read_strings, find_image_directory, find_csv
DATA_ROOT = (Path("/kaggle/input/datasets/leocaoprojects/enge707-2026-s2")
             if Path("/kaggle").exists() else PROJECT_ROOT / "data/raw")
OUTPUT_ROOT = (Path("/kaggle/working/reproduction") if Path("/kaggle").exists()
               else PROJECT_ROOT / "outputs/reproduction")
TRAIN_IMAGE_DIRECTORY = None  # Optional existing cache or explicit train/train directory.
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
RUN_TRAINING = False
RUN_TEST_PREDICTION = False
RUN_EVALUATION = False
split, mapping = load_metadata(PROJECT_ROOT)
print("Project:", PROJECT_ROOT, "Device:", DEVICE)
print(split["split"].value_counts().to_dict())

In [ ]:
from src.analysis import audit_training_images, duplicate_memberships
import matplotlib.pyplot as plt
RUN_AUDIT = False  # A full image audit is optional and is not repeated during ordinary setup.
manifest = read_strings(PROJECT_ROOT / "data/processed/training_manifest.csv")
assert manifest["image_id"].tolist() == split["image_id"].tolist()
assert manifest["class"].tolist() == split["class"].tolist()
counts = manifest["class"].value_counts().sort_index()
display(pd.DataFrame({"images": counts, "percent": counts / len(manifest) * 100}))
print("Decode status:", manifest["decode_status"].value_counts().to_dict())
print("Modes:", manifest["mode"].value_counts().to_dict())
dimensions = manifest[["width", "height"]].astype(int)
dimensions["aspect_ratio"] = dimensions["width"] / dimensions["height"]
display(dimensions.describe())
duplicates = duplicate_memberships(manifest)
display(duplicates.groupby("duplicate_type").agg(groups=("group_id", "nunique"), images=("image_id", "nunique")))
assert duplicates["label_status"].eq("consistent").all()
fig, ax = plt.subplots(figsize=(8, 3.5), layout="constrained")
counts.plot.bar(ax=ax, color="#7596a7", rot=30)
ax.set(xlabel="Training class", ylabel="Labelled images", title="Training class distribution")
plt.show()

## 2. Existing examples and quality evidence

The original seed-42 gallery and dimension plots are retained rather than rerunning the image audit. The manifest retains every labelled row, dimensions, original mode, decode findings and both hashes. Duplicate memberships are saved in `outputs/metrics/training_duplicate_groups.csv`. All 63,660 images decoded successfully; tiny and elongated images remain in the data.

In [ ]:
from IPython.display import Image, display
for name in ["training_gallery_seed42.png", "training_dimension_distribution.png"]:
    display(Image(filename=str(PROJECT_ROOT / "outputs/figures" / name), width=850))

In [ ]:
if RUN_AUDIT:
    labels = read_strings(find_csv(DATA_ROOT, "train_labels.csv"))
    assert labels[["image_id", "class"]].equals(manifest[["image_id", "class"]])
    directory = TRAIN_IMAGE_DIRECTORY or find_image_directory(DATA_ROOT, labels["image_id"])
    new_manifest = audit_training_images(labels, directory, OUTPUT_ROOT / "training_audit")
else:
    print("Full audit disabled; original audit evidence retained.")